# 06 â€” ML: Demand Forecasting for Fleet Rebalancing
**Predict neighbourhood-level demand to enable proactive pre-positioning**

## Industry Problem
Bike-share operators lose revenue and rider trust when stations run empty
(missed rides) or full (failed returns). The solution is **predictive
rebalancing** â€” moving bikes *before* imbalances happen.

## Approach
| Phase | Condition | Method |
|-------|-----------|--------|
| **Cold-start** | < 48 hourly data points | Domain-prior curves Ã— observed station profile |
| **Data-driven** | â‰¥ 48 hourly data points | Random Forest on observed demand with temporal + spatial features |

### Target Variable
`predicted_demand` = **demand pressure index** per neighbourhood per hour,
derived from observed utilization shifts (Î”No_Bikes) and critical events.
A higher value means more bikes are being consumed and rebalancing is needed.

### Feature Engineering
- Cyclical hour/day encoding (sin/cos â€” captures 24 h and 7 d periodicity)
- Rush-hour / weekend binary flags
- Lag features: demand at t-1, t-2, t-3 hours
- Neighbourhood profile: station count, total capacity, density tier

### Confidence Intervals
Real quantile-based intervals from the Random Forest tree ensemble
(10th / 90th percentile of tree predictions), not crude `Â±RMSE` bands.

### Output
Gold table **`forecast_demand`** (14 columns) â†’ Direct Lake â†’ Power BI
+ feeds NB07 Activator Alerts for proactive pre-positioning.

### Prerequisites
1. Attach **bicycles_gold** as the default lakehouse
2. Add **bicycles_silver** as an additional lakehouse
3. Run **NB03 â†’ NB04** first so Silver/Gold tables exist

In [ ]:
# ============================================================
# CELL 1 â€” Configuration Â· Load Training Data
# ============================================================

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import (
    col, lit, when, coalesce, round as spark_round,
    count, avg as spark_avg, sum as spark_sum,
    min as spark_min, max as spark_max, stddev,
    hour, dayofweek, month, date_format,
    lag, current_timestamp, expr, percentile_approx,
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType, LongType,
)
from datetime import datetime, timedelta
import math

spark = SparkSession.builder.getOrCreate()

# â”€â”€ Configuration â€” use fully qualified table names for pipeline execution â”€â”€
# Silver and Gold are NON-SCHEMA lakehouses (no .dbo prefix)
SILVER_LAKEHOUSE = "bicycles_silver"
SILVER_SCHEMA    = f"{SILVER_LAKEHOUSE}"          # No dbo schema

GOLD_LAKEHOUSE   = "bicycles_gold"
GOLD_SCHEMA      = f"{GOLD_LAKEHOUSE}"            # No dbo schema

print(f"Spark version: {spark.version}")
print(f"Processing started at: {datetime.now()}")
print(f"\nSource: {SILVER_SCHEMA}")
print(f"Target: {GOLD_SCHEMA}")

# â”€â”€ Verify Silver tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"\nSilver tables ({SILVER_SCHEMA}):")
try:
    spark.sql(f"SHOW TABLES IN {SILVER_SCHEMA}").show(truncate=False)
except Exception as e:
    print(f"  Could not list tables: {e}")

# â”€â”€ Read Silver tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def read_silver(table_name):
    return spark.sql(f"SELECT * FROM {SILVER_SCHEMA}.{table_name}")

# â”€â”€ Load Silver tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df_hourly  = read_silver("silver_hourly_demand")
df_stations = read_silver("silver_station_profile")
df_neighbourhood = read_silver("silver_neighbourhood_metrics")

# Weather-enriched demand (from NB03a) â€” used for weather-aware features
try:
    df_weather_demand = read_silver("silver_weather_bike_joined")
    WEATHER_AVAILABLE = df_weather_demand.count() > 0
except Exception:
    WEATHER_AVAILABLE = False
    df_weather_demand = None

# â”€â”€ Determine training path â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
hourly_count = df_hourly.count()
DATA_DRIVEN  = hourly_count >= 48

print(f"Hourly demand rows : {hourly_count:,}")
print(f"Weather available  : {WEATHER_AVAILABLE}")
print(f"Training mode      : {'Data-driven RF' if DATA_DRIVEN else 'Cold-start domain priors'}")
print(f"Stations           : {df_stations.count():,}")
print(f"Neighbourhoods     : {df_neighbourhood.count():,}")

In [ ]:
# ============================================================
# CELL 2 â€” Feature Engineering + Neighbourhood Lookups
# ============================================================
# Build a feature matrix from silver_hourly_demand.
# Target: "demand" â€” demand pressure index derived from:
#   event_count (activity) weighted by critical events and rebalance
#   triggers, so higher values = more bikes consumed / more stress.
#
# Also build lookup dicts needed by forecast generation (Cell 5):
#   all_neighbourhoods  â€” list of neighbourhood names
#   hood_profiles       â€” {hood: {station_count, total_capacity}}
#   latest_by_hood      â€” {hood: {avg_utilization, avg_bikes, ...}}
# ============================================================

# â”€â”€ Neighbourhood profile lookup (always needed) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
all_neighbourhoods = sorted(
    [r["Neighbourhood"] for r in df_neighbourhood.select("Neighbourhood").distinct().collect()]
)

hood_profiles = {}
for row in df_neighbourhood.select("Neighbourhood", "station_count", "total_capacity").collect():
    hood_profiles[row["Neighbourhood"]] = {
        "station_count": int(row["station_count"]) if row["station_count"] else 5,
        "total_capacity": int(row["total_capacity"]) if row["total_capacity"] else 100,
    }
print(f"Neighbourhood profiles: {len(hood_profiles)} loaded")

# â”€â”€ Latest known values per neighbourhood (always needed) â”€â”€â”€â”€â”€
latest_by_hood = {}
if hourly_count > 0:
    w_latest = Window.partitionBy("Neighbourhood").orderBy(col("event_hour").desc())
    df_latest = (
        df_hourly
        .withColumn("_rn", F.row_number().over(w_latest))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )
    for row in df_latest.collect():
        latest_by_hood[row["Neighbourhood"]] = {
            "avg_utilization": float(row["avg_utilization"]) if row["avg_utilization"] else 0.5,
            "avg_bikes": float(row["avg_bikes"]) if row["avg_bikes"] else 10.0,
            "active_stations": int(row["active_stations"]) if row["active_stations"] else 5,
            "demand": float(row["event_count"]) if row["event_count"] else 0.0,
        }
print(f"Latest-known values  : {len(latest_by_hood)} neighbourhoods")

# â”€â”€ Feature matrix (data-driven path only) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if DATA_DRIVEN:
    # â”€â”€ Choose data source: weather-enriched if available â”€â”€â”€â”€â”€
    if WEATHER_AVAILABLE:
        df_base = df_weather_demand
        print("  Using weather-enriched demand (silver_weather_bike_joined)")
    else:
        df_base = df_hourly
        print("  Using base demand (silver_hourly_demand) â€” no weather features")

    # â”€â”€ Target: demand pressure index â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    df_features = (
        df_base
        .withColumn(
            "demand",
            spark_round(
                col("event_count")
                + col("critical_events") * 2.0
                + col("rebalance_triggers") * 1.5
                + col("demand_spike_count") * 1.0,
                1,
            ),
        )
        .select(
            "event_hour", "Neighbourhood",
            "demand",
            "avg_utilization", "avg_bikes",
            "critical_events", "active_stations",
            "hour_of_day", "day_of_week", "is_rush_hour",
            # Include weather columns if present
            *([
                "temperature_c", "wind_speed_kmh", "relative_humidity",
                "cloud_cover_pct", "cycling_comfort_index",
                "has_precipitation", "weather_demand_impact",
            ] if WEATHER_AVAILABLE else []),
        )
        # â”€â”€ Cyclical time encoding â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        .withColumn("hour_sin", expr("sin(2 * pi() * hour_of_day / 24)"))
        .withColumn("hour_cos", expr("cos(2 * pi() * hour_of_day / 24)"))
        .withColumn("day_sin",  expr("sin(2 * pi() * day_of_week / 7)"))
        .withColumn("day_cos",  expr("cos(2 * pi() * day_of_week / 7)"))
        .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
        .withColumn("rush_flag",  col("is_rush_hour").cast("int"))
    )

    # â”€â”€ Cast weather booleans to int for ML â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if WEATHER_AVAILABLE:
        df_features = (
            df_features
            .withColumn("has_precip_flag", col("has_precipitation").cast("int"))
            .na.fill(0, ["temperature_c", "wind_speed_kmh", "relative_humidity",
                         "cloud_cover_pct", "cycling_comfort_index",
                         "has_precip_flag", "weather_demand_impact"])
        )

    # â”€â”€ Lag features (previous 1â€“3 hours, same neighbourhood) â”€
    w = Window.partitionBy("Neighbourhood").orderBy("event_hour")
    df_features = (
        df_features
        .withColumn("demand_lag_1h", lag("demand", 1).over(w))
        .withColumn("demand_lag_2h", lag("demand", 2).over(w))
        .withColumn("demand_lag_3h", lag("demand", 3).over(w))
        .withColumn("util_lag_1h",   lag("avg_utilization", 1).over(w))
    )

    # Drop rows where lag is null (first 3 hours of each neighbourhood)
    # instead of filling with 0 (which corrupts the signal)
    before_drop = df_features.count()
    df_features = df_features.na.drop(subset=["demand_lag_1h", "demand_lag_2h", "demand_lag_3h"])
    after_drop = df_features.count()

    # â”€â”€ Enrich with neighbourhood profile â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    hood_profile = df_neighbourhood.select(
        col("Neighbourhood").alias("_hood"),
        col("station_count").alias("hood_station_count"),
        col("total_capacity").alias("hood_total_capacity"),
    )
    df_features = df_features.join(
        hood_profile,
        df_features["Neighbourhood"] == hood_profile["_hood"],
        "left",
    ).drop("_hood").na.fill(0)

    print(f"\nFeature matrix: {after_drop:,} rows Ã— {len(df_features.columns)} columns")
    print(f"  (Dropped {before_drop - after_drop} rows with null lags)")
    print(f"  Features: {[c for c in df_features.columns if c not in ('event_hour','Neighbourhood','demand')]}")
else:
    df_features = None
    print("\nSkipping feature engineering â€” cold-start mode")

In [ ]:
# ============================================================
# CELL 3 â€” Train Model (Data-Driven) or Build Priors (Cold-Start)
# ============================================================

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

model     = None
rmse      = None
mae       = None
r2        = None
train_cnt = 0
test_cnt  = 0

# â”€â”€ Domain-prior curves (always defined â€” used by cold-start
#    and by the poor-model blend fallback in Cell 6) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# London bike-share temporal pattern (normalised 0â€“1):
# Source: TfL Santander Cycles usage data, well-documented pattern.
WEEKDAY_CURVE = {
    0: 0.05, 1: 0.02, 2: 0.01, 3: 0.01, 4: 0.02, 5: 0.08,
    6: 0.25, 7: 0.75, 8: 1.00, 9: 0.60, 10: 0.35, 11: 0.40,
    12: 0.50, 13: 0.50, 14: 0.40, 15: 0.45, 16: 0.55, 17: 0.90,
    18: 1.00, 19: 0.65, 20: 0.35, 21: 0.20, 22: 0.12, 23: 0.08,
}
WEEKEND_CURVE = {
    0: 0.05, 1: 0.03, 2: 0.02, 3: 0.01, 4: 0.01, 5: 0.03,
    6: 0.08, 7: 0.15, 8: 0.30, 9: 0.50, 10: 0.70, 11: 0.85,
    12: 0.95, 13: 1.00, 14: 0.95, 15: 0.85, 16: 0.70, 17: 0.55,
    18: 0.40, 19: 0.30, 20: 0.20, 21: 0.15, 22: 0.10, 23: 0.06,
}

# â”€â”€ Shared feature list â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
NUMERIC_FEATURES = [
    "hour_of_day", "day_of_week",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "is_weekend", "rush_flag",
    "avg_utilization", "avg_bikes", "active_stations",
    "demand_lag_1h", "demand_lag_2h", "demand_lag_3h",
    "util_lag_1h",
    "hood_station_count", "hood_total_capacity",
]

# Add weather features when available
WEATHER_FEATURES = [
    "temperature_c", "wind_speed_kmh", "relative_humidity",
    "cloud_cover_pct", "cycling_comfort_index", "has_precip_flag",
]
if WEATHER_AVAILABLE:
    NUMERIC_FEATURES = NUMERIC_FEATURES + WEATHER_FEATURES
    print(f"  Weather features added: {WEATHER_FEATURES}")

if DATA_DRIVEN:
    # â”€â”€ Index neighbourhood as categorical â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    indexer = StringIndexer(
        inputCol="Neighbourhood", outputCol="neighbourhood_idx",
        handleInvalid="keep",
    )
    ALL_FEATURES = NUMERIC_FEATURES + ["neighbourhood_idx"]

    assembler = VectorAssembler(
        inputCols=ALL_FEATURES, outputCol="features",
        handleInvalid="skip",
    )

    rf = RandomForestRegressor(
        featuresCol="features", labelCol="demand",
        predictionCol="predicted_demand",
        numTrees=200,     # more trees â†’ smoother quantile estimates
        maxDepth=8,
        minInstancesPerNode=3,
        seed=42,
    )

    pipeline = Pipeline(stages=[indexer, assembler, rf])

    # â”€â”€ Train / test split (chronological, not random) â”€â”€â”€â”€â”€â”€â”€â”€
    # Use the latest 20 % of data as test to avoid data leakage
    total = df_features.count()
    train_n = int(total * 0.8)

    # Approximate chronological split via orderBy + limit
    # Drop any stale ML output columns to make re-runs safe
    _ml_out_cols = ["predicted_demand", "features", "neighbourhood_idx", "rawPrediction", "prediction"]
    df_clean = df_features
    for _c in _ml_out_cols:
        if _c in df_features.columns:
            df_clean = df_clean.drop(_c)
    df_ordered = df_clean.orderBy("event_hour")
    train_df = df_ordered.limit(train_n)
    test_df  = df_ordered.subtract(train_df)

    train_cnt = train_df.count()
    test_cnt  = test_df.count()
    print(f"Training set : {train_cnt:,} rows (80 %)")
    print(f"Test set     : {test_cnt:,} rows (20 %)")

    # â”€â”€ Train â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print("\nTraining Random Forest (200 trees, maxDepth=8) â€¦")
    model = pipeline.fit(train_df)
    print("âœ“ Model trained")

else:
    print("âœ“ Domain-prior demand curves loaded (TfL Santander pattern)")
    print("  Curves will be scaled by neighbourhood capacity in Cell 6.")

In [ ]:
# ============================================================
# CELL 4 â€” Model Evaluation
# ============================================================

if DATA_DRIVEN and model is not None:
    # Guard: drop any stale pipeline-output columns so re-runs are safe
    _stale = ["predicted_demand", "features", "neighbourhood_idx",
              "rawPrediction", "prediction"]
    _test_clean = test_df
    for _c in _stale:
        if _c in _test_clean.columns:
            _test_clean = _test_clean.drop(_c)
    predictions = model.transform(_test_clean)

    eval_rmse = RegressionEvaluator(
        labelCol="demand", predictionCol="predicted_demand", metricName="rmse"
    )
    eval_mae = RegressionEvaluator(
        labelCol="demand", predictionCol="predicted_demand", metricName="mae"
    )
    eval_r2 = RegressionEvaluator(
        labelCol="demand", predictionCol="predicted_demand", metricName="r2"
    )

    rmse = eval_rmse.evaluate(predictions)
    mae  = eval_mae.evaluate(predictions)
    r2   = eval_r2.evaluate(predictions)

    print("â”€â”€ Test-Set Metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RÂ²   : {r2:.4f}")

    if r2 < 0.0:
        MODEL_QUALITY = "poor"
        print("\nâš  RÂ² < 0 â€” model is worse than mean baseline.")
        print("  Cold-start priors will be blended with RF predictions.")
    elif r2 < 0.3:
        MODEL_QUALITY = "fair"
        print("\nâš  RÂ² < 0.3 â€” model captures some signal but is weak.")
    elif r2 < 0.6:
        MODEL_QUALITY = "good"
        print("\nâœ“ Model quality: good")
    else:
        MODEL_QUALITY = "excellent"
        print("\nâœ“ Model quality: excellent")

    # â”€â”€ Feature importance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    rf_model = model.stages[-1]  # the fitted RandomForestRegressionModel
    importances = rf_model.featureImportances.toArray()
    ALL_FEATURES = NUMERIC_FEATURES + ["neighbourhood_idx"]
    fi = sorted(
        zip(ALL_FEATURES, importances), key=lambda x: x[1], reverse=True,
    )
    print("\nâ”€â”€ Top-10 Feature Importances â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
    for feat, imp in fi[:10]:
        bar = "â–ˆ" * int(imp * 50)
        print(f"  {feat:25s} {imp:.4f}  {bar}")

else:
    MODEL_QUALITY = "cold_start"
    print("â”€â”€ Cold-Start Mode â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
    print("  No trained model â€” forecasts will use TfL domain priors")
    print("  scaled by neighbourhood station count and capacity.")
    print(f"  Neighbourhoods available: {len(all_neighbourhoods)}")
    print(f"  Model quality tag      : {MODEL_QUALITY}")

In [ ]:
# ============================================================
# CELL 5 â€” Generate 24-Hour Forecast Feature Rows
# ============================================================

import math
from datetime import datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType,
)

now = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
forecast_hours = [now + timedelta(hours=h) for h in range(1, 25)]

# Build future feature rows for every (neighbourhood, hour) combo
future_rows = []
for ts in forecast_hours:
    h  = ts.hour
    dw = ts.weekday()               # 0=Mon â€¦ 6=Sun
    is_wknd  = 1 if dw >= 5 else 0
    is_rush  = 1 if h in (7, 8, 9, 17, 18, 19) else 0
    h_sin = math.sin(2 * math.pi * h / 24)
    h_cos = math.cos(2 * math.pi * h / 24)
    d_sin = math.sin(2 * math.pi * dw / 7)
    d_cos = math.cos(2 * math.pi * dw / 7)

    for hood in all_neighbourhoods:
        # Neighbourhood profile defaults
        st_cnt = hood_profiles.get(hood, {}).get("station_count", 5)
        tot_cap = hood_profiles.get(hood, {}).get("total_capacity", 100)

        # Latest known values (or sensible defaults)
        latest = latest_by_hood.get(hood, {})
        avg_util    = latest.get("avg_utilization", 0.5)
        avg_bikes   = latest.get("avg_bikes", tot_cap * 0.5)
        active_st   = latest.get("active_stations", st_cnt)
        demand_1    = latest.get("demand", 0.0)
        demand_2    = latest.get("demand", 0.0)
        demand_3    = latest.get("demand", 0.0)
        util_1      = latest.get("avg_utilization", 0.5)

        row = (
            ts, hood, h, dw,
            h_sin, h_cos, d_sin, d_cos,
            is_wknd, is_rush,
            float(avg_util), float(avg_bikes), int(active_st),
            float(demand_1), float(demand_2), float(demand_3),
            float(util_1),
            int(st_cnt), int(tot_cap),
        )

        # Add weather defaults if weather features are in use
        if WEATHER_AVAILABLE:
            # Use seasonal defaults: ~15Â°C, no precip, 10 km/h wind, 60% humidity, 50% cloud, comfort 70
            row = row + (15.0, 10.0, 60.0, 50, 70.0, 0)

        future_rows.append(row)

future_schema_fields = [
    StructField("forecast_hour", TimestampType()),
    StructField("Neighbourhood", StringType()),
    StructField("hour_of_day", IntegerType()),
    StructField("day_of_week", IntegerType()),
    StructField("hour_sin", DoubleType()),
    StructField("hour_cos", DoubleType()),
    StructField("day_sin", DoubleType()),
    StructField("day_cos", DoubleType()),
    StructField("is_weekend", IntegerType()),
    StructField("rush_flag", IntegerType()),
    StructField("avg_utilization", DoubleType()),
    StructField("avg_bikes", DoubleType()),
    StructField("active_stations", IntegerType()),
    StructField("demand_lag_1h", DoubleType()),
    StructField("demand_lag_2h", DoubleType()),
    StructField("demand_lag_3h", DoubleType()),
    StructField("util_lag_1h", DoubleType()),
    StructField("hood_station_count", IntegerType()),
    StructField("hood_total_capacity", IntegerType()),
]

if WEATHER_AVAILABLE:
    future_schema_fields.extend([
        StructField("temperature_c", DoubleType()),
        StructField("wind_speed_kmh", DoubleType()),
        StructField("relative_humidity", DoubleType()),
        StructField("cloud_cover_pct", IntegerType()),
        StructField("cycling_comfort_index", DoubleType()),
        StructField("has_precip_flag", IntegerType()),
    ])

future_schema = StructType(future_schema_fields)

df_future = spark.createDataFrame(future_rows, schema=future_schema)

# Cache the latest-known values dict for cold-start reference
# (already populated in Cell 2 as latest_by_hood)

print(f"âœ“ Generated {df_future.count():,} forecast feature rows")
print(f"  {len(all_neighbourhoods)} neighbourhoods Ã— {len(forecast_hours)} hours")
print(f"  Forecast window: {forecast_hours[0]} â†’ {forecast_hours[-1]}")

In [ ]:
# ============================================================
# CELL 6 â€” Score Forecast + Write forecast_demand to Gold
# ============================================================
# Confidence intervals:
#   Data-driven  â†’ quantile estimates from individual RF trees
#   Cold-start   â†’ Â±25 % of predicted value (industry heuristic)
#
# Output columns MUST match the TMDL schema exactly (14 cols).
# ============================================================

from pyspark.sql.functions import (
    col, lit, when, current_timestamp, udf, array, sort_array,
    xxhash64, date_format,
)
from pyspark.sql.types import DoubleType, LongType

# â”€â”€ Helper: get per-tree predictions for quantile intervals â”€â”€
def _tree_quantile_bounds(model_stage, df_assembled, lower_q=0.10, upper_q=0.90):
    """
    Extract individual tree predictions from an RF ensemble to build
    real quantile-based confidence intervals (not fake Â±RMSE bands).

    Production-safe approach:
    - Samples up to 30 trees (statistically sufficient for quantiles)
    - Uses iterative transform-and-rename to stay within one plan
    - Handles pre-existing ensemble prediction column
    """
    trees = model_stage.trees
    n_trees = len(trees)
    pred_col = model_stage.getOrDefault("predictionCol")  # "predicted_demand"

    # Sample up to 30 trees for quantile estimation (avoids deep plans)
    max_sample = min(30, n_trees)
    step = max(1, n_trees // max_sample)
    sampled_trees = [trees[i] for i in range(0, n_trees, step)][:max_sample]
    n_sample = len(sampled_trees)

    # Save and remove ensemble prediction to avoid column collision
    df_work = df_assembled
    if pred_col in df_work.columns:
        df_work = df_work.withColumnRenamed(pred_col, "_ensemble_pred")

    # Iterative: each tree transforms â†’ creates pred_col â†’ rename to _t{i}
    # This keeps all columns in the same plan lineage (no cross-plan refs)
    for i, tree in enumerate(sampled_trees):
        df_work = tree.transform(df_work)
        df_work = df_work.withColumnRenamed(pred_col, f"_t{i}")

    tree_cols = [f"_t{i}" for i in range(n_sample)]

    # Compute quantiles row-by-row using array + sort_array
    df_work = df_work.withColumn(
        "_tree_preds", array(*[col(c) for c in tree_cols])
    )
    df_work = df_work.withColumn(
        "_sorted", sort_array("_tree_preds")
    )
    lo_idx = max(0, int(n_sample * lower_q) - 1)
    hi_idx = min(n_sample - 1, int(n_sample * upper_q))

    df_work = (
        df_work
        .withColumn("demand_lower_bound", col("_sorted").getItem(lo_idx))
        .withColumn("demand_upper_bound", col("_sorted").getItem(hi_idx))
    )

    # Restore ensemble prediction and clean up temp columns
    if "_ensemble_pred" in [f.name for f in df_work.schema.fields]:
        df_work = df_work.withColumnRenamed("_ensemble_pred", pred_col)
    drop_cols = tree_cols + ["_tree_preds", "_sorted"]
    df_work = df_work.drop(*drop_cols)
    return df_work


if DATA_DRIVEN and model is not None:
    # â”€â”€ Data-driven scoring â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Guard: drop any stale pipeline-output columns so re-runs are safe
    _stale = ["predicted_demand", "features", "neighbourhood_idx",
              "rawPrediction", "prediction"]
    _future_clean = df_future
    for _c in _stale:
        if _c in _future_clean.columns:
            _future_clean = _future_clean.drop(_c)
    # Apply the full pipeline (StringIndexer + VectorAssembler + RF)
    df_scored = model.transform(_future_clean)

    # Extract RF model stage for tree-level quantile bounds
    rf_model = model.stages[-1]
    df_scored = _tree_quantile_bounds(rf_model, df_scored)

    # Clamp predictions to non-negative
    df_scored = (
        df_scored
        .withColumn("predicted_demand",
            when(col("predicted_demand") < 0, 0.0).otherwise(col("predicted_demand")))
        .withColumn("demand_lower_bound",
            when(col("demand_lower_bound") < 0, 0.0).otherwise(col("demand_lower_bound")))
        .withColumn("demand_upper_bound",
            when(col("demand_upper_bound") < 0, 0.0).otherwise(col("demand_upper_bound")))
    )

    # If RÂ² was poor, blend RF predictions with cold-start priors
    if MODEL_QUALITY == "poor":
        WEEKDAY_CURVE_BC = spark.sparkContext.broadcast(WEEKDAY_CURVE)
        WEEKEND_CURVE_BC = spark.sparkContext.broadcast(WEEKEND_CURVE)

        @udf(DoubleType())
        def prior_demand(hour_of_day, is_weekend, hood_total_capacity):
            curve = WEEKEND_CURVE_BC.value if is_weekend == 1 else WEEKDAY_CURVE_BC.value
            return float(curve.get(hour_of_day, 0.3)) * (hood_total_capacity / 20.0)

        df_scored = (
            df_scored
            .withColumn("_prior",
                prior_demand("hour_of_day", "is_weekend", "hood_total_capacity"))
            .withColumn("predicted_demand",
                (col("predicted_demand") * 0.5 + col("_prior") * 0.5))
            .drop("_prior")
        )

else:
    # â”€â”€ Cold-start scoring â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    WEEKDAY_CURVE_BC = spark.sparkContext.broadcast(WEEKDAY_CURVE)
    WEEKEND_CURVE_BC = spark.sparkContext.broadcast(WEEKEND_CURVE)

    @udf(DoubleType())
    def cold_start_demand(hour_of_day, is_weekend, hood_total_capacity):
        curve = WEEKEND_CURVE_BC.value if is_weekend == 1 else WEEKDAY_CURVE_BC.value
        base = float(curve.get(hour_of_day, 0.3))
        # Scale by neighbourhood capacity â€” larger areas attract more riders
        return base * (hood_total_capacity / 20.0)

    df_scored = (
        df_future
        .withColumn("predicted_demand",
            cold_start_demand("hour_of_day", "is_weekend", "hood_total_capacity"))
        .withColumn("demand_lower_bound", col("predicted_demand") * 0.75)
        .withColumn("demand_upper_bound", col("predicted_demand") * 1.25)
    )


# â”€â”€ Dynamic thresholds from the prediction distribution â”€â”€â”€â”€â”€â”€â”€
# Instead of hardcoded magic numbers, use percentiles of the
# actual forecast values to set demand tier boundaries.
quantiles = df_scored.approxQuantile(
    "predicted_demand", [0.25, 0.50, 0.75], 0.05,
)
Q1, Q2, Q3 = quantiles[0], quantiles[1], quantiles[2]
print(f"Demand distribution â€” Q1: {Q1:.2f}  Median: {Q2:.2f}  Q3: {Q3:.2f}")

# â”€â”€ Build final forecast table matching TMDL schema â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df_forecast = (
    df_scored
    .withColumn("neighbourhood", col("Neighbourhood"))
    .withColumn("is_rush_hour", col("rush_flag").cast("boolean"))
    .withColumn("is_weekend_bool", col("is_weekend").cast("boolean"))
    .withColumn(
        "demand_tier",
        when(col("predicted_demand") >= Q3, "High")
        .when(col("predicted_demand") >= Q2, "Medium")
        .when(col("predicted_demand") >= Q1, "Low")
        .otherwise("Minimal"),
    )
    .withColumn(
        "pre_position_recommended",
        (col("predicted_demand") >= Q3).cast("boolean"),
    )
    .withColumn("forecast_generated_at", current_timestamp())
    .withColumn("model_quality", lit(MODEL_QUALITY))
    .select(
        xxhash64("neighbourhood", "forecast_hour").cast(LongType()).alias("forecast_key"),
        col("forecast_hour").cast("timestamp"),
        col("neighbourhood").cast("string"),
        xxhash64("Neighbourhood").cast(LongType()).alias("neighbourhood_key"),
        col("hour_of_day").cast(LongType()).alias("time_key"),
        date_format(col("forecast_hour"), "yyyyMMdd").cast(LongType()).alias("date_key"),
        col("hour_of_day").cast("int"),
        col("day_of_week").cast("int"),
        col("is_rush_hour"),
        col("is_weekend_bool").alias("is_weekend"),
        col("predicted_demand").cast("double"),
        col("demand_lower_bound").cast("double"),
        col("demand_upper_bound").cast("double"),
        col("demand_tier").cast("string"),
        col("pre_position_recommended"),
        col("forecast_generated_at").cast("timestamp"),
        col("model_quality").cast("string"),
    )
)
# â”€â”€ Write to Gold lakehouse â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df_forecast.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.forecast_demand")

written = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.forecast_demand").collect()[0]["c"]
print(f"\nâœ“ Wrote {written:,} rows â†’ Gold / forecast_demand")
print(f"  Model quality : {MODEL_QUALITY}")
print(f"  Demand tiers  : Minimal < {Q1:.1f} < Low < {Q2:.1f} < Medium < {Q3:.1f} < High")
df_forecast.groupBy("demand_tier").count().orderBy("count", ascending=False).show()

In [ ]:
# ============================================================
# CELL 7 â€” Forecast Insights & Pre-Positioning Recommendations
# ============================================================

from pyspark.sql.functions import (
    sum as _sum, avg as _avg, max as _max, min as _min,
    count, desc,
)

print("=" * 60)
print("  24-HOUR DEMAND FORECAST â€” EXECUTIVE SUMMARY")
print("=" * 60)

# â”€â”€ Peak demand hours â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
peak_hours = (
    df_forecast
    .groupBy("hour_of_day")
    .agg(_sum("predicted_demand").alias("total_demand"))
    .orderBy(desc("total_demand"))
)
print("\nðŸ“ˆ Peak Demand Hours (top 5):")
peak_hours.show(5, truncate=False)

# â”€â”€ Neighbourhoods requiring pre-positioning â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
prepos = (
    df_forecast
    .filter(col("pre_position_recommended") == True)
    .groupBy("neighbourhood")
    .agg(
        count("*").alias("prepos_hours"),
        _avg("predicted_demand").alias("avg_demand"),
        _max("predicted_demand").alias("peak_demand"),
    )
    .orderBy(desc("prepos_hours"))
)
print("\nðŸšš Pre-Positioning Recommendations by Neighbourhood:")
prepos.show(20, truncate=False)

# â”€â”€ Rush hour vs off-peak comparison â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rush_stats = (
    df_forecast
    .groupBy("is_rush_hour")
    .agg(
        _avg("predicted_demand").alias("avg_demand"),
        _sum("predicted_demand").alias("total_demand"),
    )
)
print("\nâ° Rush Hour vs Off-Peak:")
rush_stats.show()

# â”€â”€ Weekend vs weekday â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
wknd_stats = (
    df_forecast
    .groupBy("is_weekend")
    .agg(
        _avg("predicted_demand").alias("avg_demand"),
        count("*").alias("forecast_rows"),
    )
)
print("ðŸ“… Weekend vs Weekday:")
wknd_stats.show()

# â”€â”€ Confidence interval width â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ci_stats = df_forecast.select(
    _avg(col("demand_upper_bound") - col("demand_lower_bound")).alias("avg_ci_width"),
    _max(col("demand_upper_bound") - col("demand_lower_bound")).alias("max_ci_width"),
    _min(col("demand_upper_bound") - col("demand_lower_bound")).alias("min_ci_width"),
)
print("ðŸ“Š Confidence Interval Statistics:")
ci_stats.show()

# â”€â”€ Demand tier breakdown â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tier_counts = (
    df_forecast
    .groupBy("demand_tier")
    .agg(count("*").alias("hours"))
    .orderBy(desc("hours"))
)
print("ðŸ·ï¸  Demand Tier Distribution:")
tier_counts.show()

# â”€â”€ Actionable summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
total_prepos = df_forecast.filter(col("pre_position_recommended") == True).count()
total_high   = df_forecast.filter(col("demand_tier") == "High").count()
total_rows   = df_forecast.count()

print("=" * 60)
print("  DISPATCH SUMMARY")
print("=" * 60)
print(f"  Total forecast points     : {total_rows:,}")
print(f"  High-demand (hoodÃ—hour)   : {total_high:,}  ({100*total_high/max(total_rows,1):.0f} %)")
print(f"  Pre-position recommended  : {total_prepos:,}  ({100*total_prepos/max(total_rows,1):.0f} %)")
print(f"  Model quality             : {MODEL_QUALITY}")
print(f"  Forecast mode             : {'Data-driven RF' if DATA_DRIVEN else 'Cold-start domain priors'}")
print("=" * 60)
print("\nâœ… NB06 complete â€” forecast_demand table ready for:")
print("   â€¢ NB07 Activator Alerts (rebalancing triggers)")
print("   â€¢ Semantic Model / Power BI (demand dashboards)")
print("   â€¢ Dispatcher decision support (pre-positioning)")